# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print summary
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets and their fields, using @id for referencing
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    # fallback for .recordSet (possible older croissant schema version or typo)
    record_sets = getattr(metadata, 'recordSet', None)

if record_sets is None or len(record_sets) == 0:
    # fallback: discover record sets programmatically from dataset
    record_sets = []
    # Try automatic discover
    if hasattr(dataset, 'record_sets'):
        record_sets_dict = dataset.record_sets
        record_set_ids = list(record_sets_dict.keys())
    else:
        # With latest mlcroissant the simplest is to look for dataset.record_sets
        record_set_ids = []  # may not be accessible
else:
    # attribute is list of objects
    record_set_ids = [rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', str(rs)) for rs in record_sets]

if not record_set_ids:
    # Try to detect at runtime
    import warnings
    warnings.warn("No record sets detected in metadata. Attempting to enumerate all record_set @ids available from the Croissant schema file.")
    # Download and inspect the Croissant schema for record sets
    import requests
    croissant_schema = requests.get(croissant_url).json()
    if 'recordSet' in croissant_schema:
        rs_list = croissant_schema['recordSet']
        if isinstance(rs_list, (list, tuple)):
            for rs in rs_list:
                if isinstance(rs, dict) and '@id' in rs:
                    record_set_ids.append(rs['@id'])
                elif isinstance(rs, str):
                    record_set_ids.append(rs)
    elif '@graph' in croissant_schema:
        # Fallback: extract all objects of type RecordSet from @graph
        for entry in croissant_schema['@graph']:
            if entry.get('@type', '').endswith('RecordSet'):
                if '@id' in entry:
                    record_set_ids.append(entry['@id'])
    # If still none, try to list available data distributions as record sets
    if not record_set_ids and 'distribution' in croissant_schema:
        for dist in croissant_schema['distribution']:
            if isinstance(dist, dict) and '@id' in dist:
                record_set_ids.append(dist['@id'])

if not record_set_ids:
    raise RuntimeError("No record sets found in the dataset metadata or schema!")

# For each record set, identify and print its fields (by @id)
print('Available Record Sets and their Fields:')

croissant_schema = None
if 'croissant_schema' not in locals():
    import requests
    croissant_schema = requests.get(croissant_url).json()

recordset_fields_map = dict()

for rs_id in record_set_ids:
    # Find record set definition
    rs_def = None
    # Try in 'recordSet' property
    if 'recordSet' in croissant_schema:
        for rs in croissant_schema['recordSet']:
            rid = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
            if rid == rs_id:
                rs_def = rs
                break
    # If not found, try @graph
    if rs_def is None and '@graph' in croissant_schema:
        for entry in croissant_schema['@graph']:
            if entry.get('@id', None) == rs_id:
                rs_def = entry
                break
    # Extract field @ids
    field_ids = []
    if rs_def and 'field' in rs_def:
        fields = rs_def['field']
        if isinstance(fields, list):
            for field in fields:
                if isinstance(field, dict) and '@id' in field:
                    field_ids.append(field['@id'])
                elif isinstance(field, str):
                    field_ids.append(field)
    # Print
    print(f"  Record set @id: {rs_id}")
    if field_ids:
        print(f"    Fields (@id):")
        for fid in field_ids:
            print(f"      - {fid}")
    else:
        print("    (No fields listed in schema for this record set)")
    recordset_fields_map[rs_id] = field_ids

## 3. Data Extraction
Load data from each available record set into a DataFrame for further analysis.

In [ ]:
# Extract data from all available record sets
# Record set @ids discovered in previous step
dataframes = {}
for record_set_id in recordset_fields_map.keys():
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records and isinstance(records, list):
            df = pd.DataFrame(records)
        else:
            df = pd.DataFrame()
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# For demonstration, pick the largest DataFrame as main (or the first one)
if dataframes:
    main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[0])
else:
    raise RuntimeError("No dataframes loaded from dataset!")

main_df = dataframes[main_record_set_id]
print(f"\nFields (@id) in main DataFrame ({main_record_set_id}):")
print(main_df.columns.tolist())
print("\nPreview of main DataFrame:")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, or grouping by attributes. All field references are by field `@id`.

In [ ]:
# EDA: choose a numeric field (by @id) from the main record set
cols = list(main_df.columns)
# Try to pick a likely numeric field by heuristics
import re
numeric_field_candidates = [c for c in cols if re.search(r'(age|interval|score|count|number|msi|duration|metastasis)', c, re.I)]

numeric_field_id = None
for c in numeric_field_candidates:
    if pd.api.types.is_numeric_dtype(main_df[c]):
        numeric_field_id = c
        break
if not numeric_field_id:
    # Fallback: first numeric column
    for c in cols:
        if pd.api.types.is_numeric_dtype(main_df[c]):
            numeric_field_id = c
            break
if not numeric_field_id:
    raise RuntimeError("No numeric field found in main DataFrame.")

# Show statistics before filtering
print(f"Statistics for '{numeric_field_id}':")
print(main_df[numeric_field_id].describe())

# Set a threshold for illustrative purposes (e.g., above median)
threshold = main_df[numeric_field_id].median()
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"\nFiltered records where {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Pick a group field for aggregation — choose a categorical field
group_field_candidates = [c for c in cols if re.search(r'(sex|gender|site|location|status|type|category|group|biomarker)', c, re.I)]
group_field = None
for c in group_field_candidates:
    if pd.api.types.is_object_dtype(main_df[c]) or main_df[c].dtype.name == 'category':
        group_field = c
        break

if group_field:
    grouped_stats = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} grouped by {group_field}:")
    print(grouped_stats.head())
else:
    print("\nNo suitable group field found for aggregation.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (before normalization)
plt.figure(figsize=(8,5))
sns.histplot(main_df[numeric_field_id].dropna(), bins=10, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If group_field exists, boxplot of numeric field by group_field
if group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=main_df, x=group_field, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we used the `mlcroissant` library to load and explore the [Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) dataset. Through record set and field `@id` referencing, we demonstrated data extraction, basic EDA, normalization, grouping, and visualization. This framework facilitates robust, reproducible biomedical data exploration following FAIR principles.